In [2]:
import numpy as np
from deap import base, creator, tools, algorithms
import cvxpy as cp
from scipy.signal import cont2discrete , tf2ss
import control as ct

# Define system matrices
# A = np.array([[1, 0.5, -0.125], [0, 1, -0.5], [0, 0, 1]])
# B = np.array([[-0.0208], [-0.125], [0.5]])
T_eng = 0.460
K_eng = 0.732
A_f = -1/T_eng
B_f = -K_eng/T_eng
C_f = np.eye(3)
T_hw = 1.6
Ts = 0.05
T_total = 10
T = int(T_total/Ts)
v0 = 15           # Initial target velocity
init_dist = 5     # Initial distance between vehicles

# Discretize the system
A = np.array([[0, 1, -T_hw], [0, 0, -1], [0, 0, A_f]])
B = np.array([[0], [0], [B_f]])
sys2 = cont2discrete((A, B, np.eye(3), 0), Ts, method='zoh')
A, B, C, D , dt = sys2

# Define initial and reference states
x0 = np.array([100, 16.0, 0])
xr = np.array([8.0, 0, 0])

# Define constraints
umax = 0.3
umin = -0.5
xmin = np.array([[3], [-50], [-4.5]])
xmax = np.array([[600], [50.0], [3.0]])

print(xmin.size)

# Define GA parameters
population_size = 100
generations = 35
crossover_prob = 0.7
mutation_prob = 0.3

# Define fitness function
def fitness(weights):
    # Define MPC parameters
    N = 10  # prediction horizon
    nx = A.shape[0]  # number of states
    nu = B.shape[1]  # number of inputs

    # Define decision variables
    x = cp.Variable((nx, N+1))
    u = cp.Variable((nu, N))

    # Define cost function
    Q = np.diag(weights[:3])

    x0 = np.array([100, 16.0, 0])
    xr = np.array([8.0, 0, 0])
    w1 = weights[0]
    w2 = weights[1]
    w3 = weights[2]
    R = np.eye(nu)*weights[3]
   
    # cost = 0
    # for t in range(N):
    #     cost +=  (w1*((x[1, t]-xr[1])/10) + w2*(x[2, t]-xr[2]) + w3*((x[0, t]-xr[0])/100)) #cvxpy.quad_form(x[:, t]- xr, Q) #
    #     cost += 0.5*weights[3]*u[:, t] # cvxpy.quad_form(u[:, t], R)

    # cost += 0.5 *((x[1, N]-xr[1]) + (x[2, N]-xr[2]) + (x[0, N]-xr[0]))  # terminal cost
    # constraints = []
    
    # for k in range(N):
    #     constraints += [x[:,k+1] == A@x[:,k] + B@u[:,k], umin <= u[:,k], u[:,k] <= umax]
                         
    # constraints += [x[:, N] >= xmin[:, 0], x[:,N] <= xmax[:, 0]]

    # # Define optimization problem
    # prob = cp.Problem(cp.Minimize(cost), constraints)
    costlist = 0.0
    constraints = []

    for t in range(N):
        costlist +=  (w1*((x[1, t]-xr[1])/10) + w2*(x[2, t]-xr[2]) + w3*((x[0, t]-xr[0])/100)) #cvxpy.quad_form(x[:, t]- xr, Q) #
        costlist += 0.5*weights[3]*u[:, t]
        # costlist +=  cvxpy.quad_form(x[:, t] - xr[1], Q)
        # # w1*(x[1, t]-xr[1]) # + w2*(x[2, t]-xr[2]) + w3*((x[0, t]-xr[0]))) #(w1*((x[1, t]-xr[1]))**2 + w2*(x[2, t]-xr[2])**2 + w3*((x[0, t]-xr[0]))**2)
        # costlist +=  cvxpy.quad_form(u[:, t], R)  # 0.5*u[:, t]
        #         # 0.5 * cvxpy.quad_form(u[:, t], R)

        constraints += [x[:, t + 1] == A * x[:, t] + B * u[:, t]]

        if xmin is not None:
            constraints += [x[:, t] >= xmin[:, 0]]  # state is greater than x min 
        if xmax is not None:
            constraints += [x[:, t] <= xmax[:, 0]] # state is less than x max 

    # costlist += 0.5 *((x[1, N]-xr[1]) + (x[2, N]-xr[2]) + (x[0, N]-xr[0])) # terminal cost #0.5 *((x[1, N]-xr[1])**2 + (x[2, N]-xr[2])**2 + (x[0, N]-xr[0])**2)  # terminal cost
    costlist += 0.5 *((x[1, N]-xr[1]) + (x[2, N]-xr[2]) + (x[0, N]-xr[0]))
    if xmin is not None:
        constraints += [x[:, N] >= xmin[:, 0]]
    if xmax is not None:
        constraints += [x[:, N] <= xmax[:, 0]]

    if umax is not None:
        constraints += [u <= umax]  # input constraints
    if umin is not None:
        constraints += [u >= umin]  # input constraints

    # print("this is inside functio",x0)
    constraints += [x[:, 0] == x0]  # inital state constraints

    prob = cp.Problem(cp.Minimize(costlist), constraints)

    # Solve MPC problem for a given initial state
    u_mpc = np.zeros((nu, N))
    for i in range(N):
        prob.solve()
        u_mpc[:,i] = u[:,0].value
        x0 = A@x0 + B@u_mpc[:,i]

    # Calculate fitness as the cost of the MPC problem
    return costlist.value

def run_ga(population_size, generations, crossover_prob, mutation_prob):
    # Create genetic algorithm toolbox
    creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
    creator.create("Individual", np.ndarray, fitness=creator.FitnessMin)

    toolbox = base.Toolbox()
    toolbox.register("weights", np.random.uniform, low=-1.0, high=10.0, size=(4,))
    toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.weights)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("mate", tools.cxTwoPoint)
    toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=0.1, indpb=0.1)
    toolbox.register("select", tools.selTournament, tournsize=3)
    toolbox.register("evaluate", fitness)

    # Initialize population
    population = toolbox.population(n=population_size)

    # Evolve population
    for gen in range(generations):
        offspring = []

        # Apply crossover and mutation to population
        for i in range(0, population_size, 2):
            ind1 = population[i]
            ind2 = population[i+1]
            if np.random.random() < crossover_prob:
                offspring1, offspring2 = toolbox.mate(ind1, ind2)
                del offspring1.fitness.values
                del offspring2.fitness.values
                offspring.append(offspring1)
                offspring.append(offspring2)
            else:
                offspring.append(ind1)
                offspring.append(ind2)

        # Apply mutation to offspring
        for ind in offspring:
            if np.random.random() < mutation_prob:
                toolbox.mutate(ind)
                del ind.fitness.values

        # Evaluate fitness of offspring
        fitnesses = list(map(toolbox.evaluate, offspring))
        for ind, fit in zip(offspring, fitnesses):
            ind.fitness.values = fit

        # Replace population with offspring
        population[:] = offspring

    # Select the best individual from the final population
    best_ind = tools.selBest(population, k=1)[0]
    best_weights = best_ind

    # Print results
    print("Best weights:", best_weights)
    print("Best fitness:", fitness(best_weights))

    return best_weights

# Run genetic algorithm
best_weights = run_ga(population_size, generations, crossover_prob, mutation_prob)


3


C:\Users\harsh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.8_qbz5n2kfra8p0\LocalCache\local-packages\Python38\site-packages\deap\creator.py:138: RuntimeWarning: A class named 'FitnessMin' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
C:\Users\harsh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.8_qbz5n2kfra8p0\LocalCache\local-packages\Python38\site-packages\deap\creator.py:138: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


Best weights: [9.4551332  9.33968588 4.38513141 2.68176587]
Best fitness: [-1357.83403724]
